# SAMMD student OpenMM workflow

This notebook starts from a SAMMD YAML configuration, validates the science inputs, builds a deterministic SAMMD plan, constructs the current OpenMM prototype system, and runs a short configurable OpenMM simulation.

Status: the first half uses the stable public API (`load_config` and `build_system`). The OpenMM construction cells still use private prototype modules because the public OpenMM build facade is not final yet. Treat the output as a smoke test, not production MD or evidence of scientific convergence.

## 1. Choose the configuration and run controls

Edit `examples/student_config.yaml` to describe the surface, SAM, solvent, reactant, and thermodynamic conditions. Then adjust the short-run controls below. Keep the first run short; increase the duration only after the smoke run succeeds.

`requested_platform = "CUDA"` asks OpenMM to use the local GPU first. SAMMD's runtime helper will fall back to another available OpenMM platform if CUDA is unavailable, although CPU fallback may be much slower.

In [ ]:
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

config_path = project_root / "examples" / "student_config.yaml"
output_dir = project_root / "outputs" / "student_openmm_notebook"

duration_ns = 0.02
trajectory_frames = 10
requested_platform = "CUDA"
overwrite_outputs = True

print(config_path)
print(output_dir)

## 2. Load and validate the SAMMD YAML

`load_config()` checks the YAML with the Pydantic schema. Unknown keys and unsupported MVP choices fail here before any expensive OpenMM work starts.

In [ ]:
from sammd import load_config

config = load_config(config_path)

print(f"surface: {config.surface.metal}({config.surface.facet})")
print(f"SAM: {config.sam.components[0].name} ({config.sam.components[0].smiles})")
print(f"solvent: {config.solvent.components[0].name}")
print(f"reactant: {config.reactants[0].name} at {config.reactants[0].concentration_millimolar:g} mM")
print(f"temperature: {config.simulation.temperature_k:g} K")

## 3. Build the deterministic SAMMD plan

`build_system()` currently returns a planning object. This is the public SAMMD API boundary: it selects the Pd slab, binding sites, SAM placements, finite solution counts, and output paths.

In [ ]:
from sammd import build_system

plan = build_system(config, output_dir=output_dir, seed=config.simulation.seed)
planned_slab_path = plan.write_planned_slab_mmcif(overwrite=True)

print(f"adjusted slab size: {plan.slab.lateral_size_nm} nm")
print(f"binding sites: {len(plan.binding_sites)}")
print(f"SAM placements: {len(plan.sam_placements.placements)}")
print(f"solution counts: {plan.solution.molecule_counts}")
print(f"planned slab visualization: {planned_slab_path}")

## 4. Resolve the OpenMM run schedule

The notebook turns `duration_ns`, the YAML timestep, and the requested number of frames into OpenMM steps and reporter intervals.

In [ ]:
from sammd.workflow import prepare_outputs, resolve_run_schedule, smoke_paths

paths = smoke_paths(output_dir)
prepare_outputs(paths, overwrite=overwrite_outputs)
schedule = resolve_run_schedule(
    duration_ns=duration_ns,
    timestep_fs=config.simulation.timestep_fs,
    steps=None,
    frames=trajectory_frames,
    report_interval=None,
)

print(f"steps: {schedule.total_steps}")
print(f"timestep: {schedule.timestep_fs:g} fs")
print(f"simulated time: {schedule.simulated_duration_ns:g} ns")
print(f"report interval: {schedule.report_interval} steps")

## 5. Build the prototype OpenMM system

This cell uses private prototype modules. In a future SAMMD release, this should become a public `build_openmm_system(config)` style API. For now, keep the molecule choices simple and validate one change at a time.

In [ ]:
from sammd._openmm_backend import build_openmm_smoke_system
from sammd._openmm_templates import (
    molecule_template_from_smiles,
    require_openff_modules,
    require_openmm_modules,
)
from sammd.openmm_build import OpenMMSmokeBuilder, OpenMMSmokeBuildOptions

modules = require_openmm_modules()
openff_modules = require_openff_modules()

sam_component = config.sam.components[0]
reactant = config.reactants[0]
solvent = config.solvent.components[0]

sam_template = molecule_template_from_smiles(
    modules, openff_modules, sam_component.smiles, sam_component.name
)
reactant_template = molecule_template_from_smiles(
    modules, openff_modules, reactant.smiles, reactant.name
)
solvent_template = molecule_template_from_smiles(
    modules, openff_modules, solvent.smiles, solvent.name
)

openmm_build = (
    OpenMMSmokeBuilder.from_plan(
        modules=modules,
        plan=plan,
        construction_fn=build_openmm_smoke_system,
    )
    .add_surface()
    .add_sam_layer(sam_template)
    .add_reactants(reactant_template, count=plan.solution.reactants[0].count)
    .add_solvent(solvent_template, count=plan.solution.solvent_components[0].count)
    .finalize(
        OpenMMSmokeBuildOptions(
            sulfur_height_nm=0.18,
            solvent_padding_nm=config.solvent.padding_nm,
            packmol_working_dir=paths.packmol_dir,
            pressure_bar=config.simulation.pressure_bar,
            temperature_k=config.simulation.temperature_k,
            pd_s_sigma_nm=0.22,
            pd_s_epsilon_kcal_mol=2.0,
        )
    )
)

print(f"atoms: {openmm_build.system.getNumParticles()}")
print(f"Pd atoms: {len(openmm_build.pd_indices)}")
print(f"SAM molecules: {openmm_build.sam_count}")
print(f"reactant molecules: {openmm_build.reactant_count}")
print(f"solvent molecules: {openmm_build.solvent_count}")

## 6. Write topology and serialized system artifacts

These files make the starting system inspectable outside the notebook. `topology.cif` contains the full prototype topology and starting coordinates.

In [ ]:
from sammd.io import safe_write_text

with paths.topology.open("w", encoding="utf-8") as handle:
    modules.app.PDBxFile.writeFile(
        openmm_build.topology,
        openmm_build.positions_quantity,
        handle,
        keepIds=True,
    )

safe_write_text(
    paths.system_xml,
    modules.openmm.XmlSerializer.serialize(openmm_build.system),
    overwrite=True,
)

print(paths.topology)
print(paths.system_xml)

## 7. Run OpenMM

This run minimizes the starting coordinates, initializes velocities at the configured temperature, writes a DCD trajectory, and records thermodynamics. A very large initial potential energy can occur before minimization because the packed starting geometry can contain close contacts.

Smoke-test checklist before interpreting anything: YAML validation should pass, finite molecule counts should look plausible, OpenMM system construction should complete without missing-parameter errors, minimization should finish without NaNs, the requested steps should complete, final temperature should be near the target, and all output files should exist. Passing this checklist means the workflow runs; it does not prove equilibration or scientific convergence.

In [ ]:
import json

from sammd.openmm_runtime import create_simulation_with_platform_fallback, read_energy

selection = create_simulation_with_platform_fallback(
    openmm_build.topology,
    openmm_build.system,
    openmm_build.positions_quantity,
    platform_name=requested_platform,
    temperature_k=config.simulation.temperature_k,
    timestep_fs=config.simulation.timestep_fs,
    friction_per_ps=1.0,
    openmm_module=modules.openmm,
    app_module=modules.app,
    unit_module=modules.unit,
)
simulation = selection.simulation
simulation.reporters.append(
    modules.app.DCDReporter(
        str(paths.trajectory), schedule.report_interval, enforcePeriodicBox=True
    )
)
simulation.reporters.append(
    modules.app.StateDataReporter(
        str(paths.thermodynamics),
        schedule.report_interval,
        step=True,
        time=True,
        potentialEnergy=True,
        kineticEnergy=True,
        totalEnergy=True,
        temperature=True,
        separator=",",
    )
)

print(f"OpenMM platform: {selection.platform_name}")
initial = read_energy(simulation, include_kinetic=False, unit_module=modules.unit)
simulation.minimizeEnergy(maxIterations=0)
minimized = read_energy(simulation, include_kinetic=False, unit_module=modules.unit)
simulation.context.setVelocitiesToTemperature(
    config.simulation.temperature_k, config.simulation.seed
)
simulation.step(schedule.total_steps)
final = read_energy(simulation, include_kinetic=True, unit_module=modules.unit)

summary = {
    "platform": selection.platform_name,
    "atoms": openmm_build.system.getNumParticles(),
    "steps": schedule.total_steps,
    "simulated_duration_ns": schedule.simulated_duration_ns,
    "initial_potential_kj_mol": initial.potential_energy_kj_mol,
    "minimized_potential_kj_mol": minimized.potential_energy_kj_mol,
    "final_potential_kj_mol": final.potential_energy_kj_mol,
    "final_temperature_k": final.temperature_k,
    "topology": str(paths.topology),
    "trajectory": str(paths.trajectory),
    "thermodynamics": str(paths.thermodynamics),
}
safe_write_text(paths.summary, json.dumps(summary, indent=2) + "\n", overwrite=True)

print(f"initial potential: {initial.potential_energy_kj_mol:.3g} kJ/mol")
print(f"minimized potential: {minimized.potential_energy_kj_mol:.3g} kJ/mol")
print(f"final temperature: {final.temperature_k:.2f} K")
print(paths.summary)

## 8. Inspect the outputs

A successful smoke run should minimize without NaNs, complete all requested steps, report a final temperature near the configured target, and write the files below. This does not prove equilibration or scientific convergence.

In [ ]:
for artifact in [
    paths.summary,
    paths.topology,
    paths.trajectory,
    paths.thermodynamics,
    paths.system_xml,
]:
    print(f"{artifact}: {artifact.exists()}")

print(json.dumps(summary, indent=2))

## What to change next

Change one input at a time in `examples/student_config.yaml`, then rerun from the validation cell. Good first changes are SAM molecule identity, reactant concentration, solvent composition, temperature, or short-run duration. Ask before interpreting any trajectory as a production simulation.